# SI: Single-Dye Photobleaching

Reads a `Single_frame_database.h5` for a chosen dye, screens molecules for
single-step photobleaching, and produces a 3 × 3 panel figure.  
Intensity traces are extracted from the **raw Bayer image** (12 × 12 pixel ROI,
converted to photoelectrons) so that background levels are visible alongside
the bleaching step.

**Workflow**
1. Pre-filter molecules by minimum detected-frame count.
2. For each candidate, extract the ROI intensity trace from the raw TIFF stack.
3. Run PELT change-point detection (same algorithm as the FRET analysis).
4. Classify as single-step if exactly one downward step is found.
5. Collect the first `N_FIGURE` passing candidates and plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import ruptures as rpt
import os
import gc
from pathlib import Path
from tqdm import tqdm

import sys
sys.path.append('../../..')

from src import IOFunctions
from src import sCMOSFunctions

IO   = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()

## Camera calibration

In [ ]:
cal_dir = Path('../../../Camera_Calibrations/Ximea_Camera/')

gain    = IO.read_tiff(str(cal_dir / 'gain.tif'))
offset  = IO.read_tiff(str(cal_dir / 'offset.tif'))
rqe     = IO.read_tiff(str(cal_dir / 'rqe.tif'))

print(f'Calibration maps loaded — shape: {gain.shape}')

## Data configuration

Edit `dye_config` to point at the dyes you want to include.
Each entry maps a label to a `(Single_frame_database path, raw TIFF folder)` tuple.  
The raw folder must sit inside the same dye directory and contain the `.ome.tif` files
that were used to build the database.

In [ ]:
# SMB share — adjust if the mount path differs on your machine
SMB_BASE = Path(f"/run/user/{os.getuid()}/gvfs/"
                "smb-share:server=intelliflash-mgmt-b.ch.private.cam.ac.uk,"
                "share=sycamore_asap_server/"
                "2026_Multicolour_Paper/Data/Ximea/Single_Dye_Experiments")

dye_config = {
    'ATTO488':  (SMB_BASE / 'ATTO488/Single_frame_database.h5',
                 SMB_BASE / 'ATTO488/40mW488_488LP_BP520-44_1'),
    'ATTO514':  (SMB_BASE / 'ATTO514/Single_frame_database.h5',
                 SMB_BASE / 'ATTO514/30p_515_LP515_BP540_80_1'),
    'ATTO520':  (SMB_BASE / 'ATTO520/Single_frame_database.h5',
                 SMB_BASE / 'ATTO520/30p_515_LP515_BP540_80_1'),
    'ATTO565':  (SMB_BASE / 'ATTO565/Single_frame_database.h5',
                 SMB_BASE / 'ATTO565/20perc_561_LP561_BP582-64_1'),
    'ATTO594':  (SMB_BASE / 'ATTO594/Single_frame_database.h5',
                 SMB_BASE / 'ATTO594/40p_NF_785SP_1_488_561dichro'),
    'ATTO620':  (SMB_BASE / 'ATTO620/Single_frame_database.h5',
                 SMB_BASE / 'ATTO620/40per_638_1_635LP_1'),
    'ATTO633':  (SMB_BASE / 'ATTO633/Single_frame_database.h5',
                 SMB_BASE / 'ATTO633/40both638_NF_785SP_1'),
    'ATTO647N': (SMB_BASE / 'ATTO647N/Single_frame_database.h5',
                 SMB_BASE / 'ATTO647N/40both638_NF_785SP_1'),
    'ATTO655':  (SMB_BASE / 'ATTO655/Single_frame_database.h5',
                 SMB_BASE / 'ATTO655/40both638_NF_785SP_1'),
    'ATTO700':  (SMB_BASE / 'ATTO700/Single_frame_database.h5',
                 SMB_BASE / 'ATTO700/40mW638_both_NF_785sp_1'),
    'ATTORho6G':(SMB_BASE / 'ATTORho6G/Single_frame_database.h5',
                 SMB_BASE / 'ATTORho6G/30p_515_LP515_1'),
}

# --- parameters ---
chosen_dye   = 'ATTO565'  # which dye to show in the figure

ROI_SIZE     = 12    # pixels; square ROI centred on localisation
MIN_FRAMES   = 8     # minimum detected frames to consider a molecule
PRE_PAD      = 15    # frames before first detection to include in trace
POST_PAD     = 30    # frames after last detection to include in trace
CP_MIN_SIZE  = 3     # PELT minimum segment size
SIGNAL_RATIO = 3.0   # on/off mean ratio required for single-step classification
MIN_ON_FRAMES = 10   # minimum length of the bright segment — rejects traces that
                     # bleach too quickly to show a clear step in the figure
N_FIGURE     = 9     # panels in the final figure
RANDOM_SEED  = 42

## Helper functions

In [ ]:
def build_file_index(raw_folder: Path):
    """Return sorted .ome.tif list and cumulative frame offsets.

    Returns
    -------
    tifs : list of Path
    cum_offsets : np.ndarray  shape (n_files+1,)
        cum_offsets[i] = global frame index of the first frame in file i.
    """
    tifs = sorted(raw_folder.glob('*.ome.tif'))
    if not tifs:
        tifs = sorted(raw_folder.glob('*.tif'))
    counts = [IO.get_num_pages_in_TIF(str(f)) for f in tifs]
    cum_offsets = np.concatenate([[0], np.cumsum(counts)])
    return tifs, cum_offsets


def extract_roi_trace(
    tifs, cum_offsets, xc, yc,
    frame_start, frame_end,
    gain_map, offset_map, rqe_map,
    roi_size=12,
):
    """Sum photoelectrons in a roi_size × roi_size box for global frames
    [frame_start, frame_end).

    Parameters
    ----------
    tifs, cum_offsets : from build_file_index
    xc, yc           : float, localisation centre in pixels
    frame_start/end  : int, global frame range (half-open)
    gain/offset/rqe  : full-frame calibration arrays

    Returns
    -------
    trace : np.ndarray shape (frame_end - frame_start,)
    """
    half = roi_size // 2
    xpx, ypx = int(round(xc)), int(round(yc))
    x0, x1 = max(0, xpx - half), xpx + half
    y0, y1 = max(0, ypx - half), ypx + half

    g_roi = gain_map[y0:y1, x0:x1]
    o_roi = offset_map[y0:y1, x0:x1]
    r_roi = rqe_map[y0:y1, x0:x1]

    n_out = frame_end - frame_start
    trace = np.zeros(n_out, dtype=np.float32)

    for fi, tif in enumerate(tifs):
        file_start = int(cum_offsets[fi])
        file_end   = int(cum_offsets[fi + 1])

        lo = max(frame_start, file_start)
        hi = min(frame_end,   file_end)
        if lo >= hi:
            continue

        local_frames = list(range(lo - file_start, hi - file_start))
        raw = IO.read_tiff(str(tif), dtype='float32', frame=local_frames)
        if raw.ndim == 2:
            raw = raw[np.newaxis]

        roi = raw[:, y0:y1, x0:x1]
        pe  = (roi - o_roi) / g_roi * r_roi  # ADU → photoelectrons
        trace[lo - frame_start : hi - frame_start] = pe.sum(axis=(1, 2))

        del raw, roi, pe
        gc.collect()

    return trace


def find_change_points(signal, min_size=3):
    """PELT change-point detection on a 1D intensity (photoelectron) trace.

    Identical to SR_Functions._find_change_points_single:
      - model='l2'  (Gaussian, detects changes in mean intensity)
      - penalty = n * sigma^2  (same formula used in the FRET pipeline)

    Sigma is estimated from the final 20 frames of the trace, which are
    expected to be background after photobleaching (POST_PAD >= 30 ensures
    there are always enough post-bleach frames).

    Returns list of CP indices; last element is always len(signal).
    """
    signal = np.asarray(signal, dtype=float)
    n = len(signal)
    noise_window = min(n, 20)
    sigma = np.nanstd(signal[-noise_window:])
    if sigma < 1e-10:
        sigma = np.nanstd(signal)
    if sigma < 1e-10:
        return [n]
    algo = rpt.Pelt(model='l2', min_size=min_size, jump=1).fit(signal)
    penalty = n * sigma**2
    return algo.predict(pen=penalty)


def segment_means(signal, cps):
    """Return an array of per-sample segment means."""
    out = np.empty(len(signal), dtype=float)
    prev = 0
    for cp in cps:
        out[prev:cp] = np.nanmean(signal[prev:cp])
        prev = cp
    return out


def is_single_step_bleach(trace, cps, signal_ratio=3.0, min_on_frames=10):
    """Return True if the trace shows exactly one change point: high then low.

    Requires:
      - exactly one internal change point (two segments only)
      - first segment mean >= signal_ratio * second segment mean  (starts high)
      - on-segment length >= min_on_frames
    """
    internal = cps[:-1]
    if len(internal) != 1:
        return False
    cp = internal[0]
    before = np.nanmean(trace[:cp])
    after  = np.nanmean(trace[cp:])
    return (cp >= min_on_frames and
            before >= signal_ratio * max(after, 1.0))


print('Helper functions defined.')

## Screen molecules for single-step photobleaching

Reads the database for `chosen_dye`, pre-filters by `MIN_FRAMES`, then iterates
through shuffled candidates — extracting the raw ROI trace and running PELT — until
`N_FIGURE` single-step bleachers are found.

In [ ]:
db_path, raw_folder = dye_config[chosen_dye]

# --- load database ---
print(f'Loading {db_path.name}  ({chosen_dye})')
db = pd.read_hdf(str(db_path))
print(f'  {len(db):,} localizations, {db["index"].nunique():,} unique molecules')

# --- per-molecule summary ---
mol = (
    db.groupby('index')
    .agg(
        n_frames    = ('frame',   'count'),
        xc_mean     = ('xc',      'mean'),
        yc_mean     = ('yc',      'mean'),
        frame_min   = ('frame',   'min'),
        frame_max   = ('frame',   'max'),
        photons_mean= ('photons', 'mean'),
    )
    .reset_index()
)

candidates = mol[mol['n_frames'] >= MIN_FRAMES].copy()
print(f'  Candidates with ≥{MIN_FRAMES} frames: {len(candidates):,}')

candidates = candidates.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# --- build raw TIFF file index ---
print(f'Indexing TIFF files in {raw_folder.name} ...')
tifs, cum_offsets = build_file_index(raw_folder)
total_frames = int(cum_offsets[-1])
print(f'  {len(tifs)} file(s), {total_frames} total frames')

# --- screen ---
found = []

for _, row in tqdm(candidates.iterrows(), total=len(candidates),
                   desc='Screening'):
    f0 = max(0,            int(row['frame_min']) - PRE_PAD)
    f1 = min(total_frames, int(row['frame_max']) + POST_PAD + 1)

    try:
        trace = extract_roi_trace(
            tifs, cum_offsets,
            row['xc_mean'], row['yc_mean'],
            f0, f1,
            gain, offset, rqe,
            roi_size=ROI_SIZE,
        )
    except Exception:
        continue

    cps = find_change_points(trace, min_size=CP_MIN_SIZE)

    if is_single_step_bleach(trace, cps,
                              signal_ratio=SIGNAL_RATIO,
                              min_on_frames=MIN_ON_FRAMES):
        found.append(dict(
            mol_index = int(row['index']),
            xc        = row['xc_mean'],
            yc        = row['yc_mean'],
            f0        = f0,
            trace     = trace,
            cps       = cps,
        ))

    if len(found) >= N_FIGURE:
        break

print(f'\nFound {len(found)} single-step photobleaching molecules.')

## 3 × 3 figure

In [ ]:
n_cols = 3
n_rows = (N_FIGURE + n_cols - 1) // n_cols

fig, axs = plt.subplots(
    n_rows, n_cols,
    figsize=(n_cols * 2.2, n_rows * 1.7),
    constrained_layout=True,
)
axs = axs.ravel()

for i, cand in enumerate(found[:N_FIGURE]):
    ax    = axs[i]
    trace = cand['trace']
    cps   = cand['cps']
    f0    = cand['f0']
    frames = np.arange(f0, f0 + len(trace))
    means  = segment_means(trace, cps)

    # raw trace
    ax.plot(frames, trace, color='#aaaaaa', lw=0.5, alpha=0.9, zorder=1)
    # segment means
    ax.plot(frames, means, color='#d40000', lw=1.1, zorder=2)
    # change-point locations
    for cp in cps[:-1]:
        ax.axvline(f0 + cp, color='#d40000', lw=0.8, ls='--', alpha=0.6, zorder=3)

    ax.set_xlim(frames[0], frames[-1])
    ax.set_ylim(bottom=0)
    ax.tick_params(labelsize=6, length=2, pad=1)

    row_idx = i // n_cols
    col_idx = i %  n_cols
    if col_idx == 0:
        ax.set_ylabel('Intensity / pe', fontsize=6)
    if row_idx == n_rows - 1:
        ax.set_xlabel('Frame', fontsize=6)

# hide unused panels
for j in range(len(found), len(axs)):
    axs[j].set_visible(False)

fig.suptitle(f'{chosen_dye} — single-step photobleaching ({ROI_SIZE}×{ROI_SIZE} px ROI)',
             fontsize=8)

fig_path = Path('../../../Papers/Multicolour/SI/SI_Single_Dye_Photobleaching.svg')
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=300, format='svg', bbox_inches='tight')
plt.show()
print(f'Saved → {fig_path}')

## (Optional) Quick-look: single molecule

Inspect one candidate interactively before committing to the figure.

In [ ]:
idx = 0  # which candidate to inspect
cand  = found[idx]
trace = cand['trace']
cps   = cand['cps']
f0    = cand['f0']
frames = np.arange(f0, f0 + len(trace))

print(f"Molecule index: {cand['mol_index']}")
print(f"Position: xc={cand['xc']:.1f}, yc={cand['yc']:.1f} px")
print(f"Frame range: {f0} – {f0 + len(trace)}")
print(f"Change points: {cps}")
print(f"Segment means: {[float(f'{np.nanmean(trace[p:q]):.0f}') for p, q in zip([0]+cps[:-1], cps)]}")

fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(frames, trace, color='#aaaaaa', lw=0.6)
ax.plot(frames, segment_means(trace, cps), color='#d40000', lw=1.2)
for cp in cps[:-1]:
    ax.axvline(f0 + cp, color='#d40000', lw=1, ls='--', alpha=0.7)
ax.set_xlabel('Frame')
ax.set_ylabel('Intensity / pe')
ax.set_ylim(bottom=0)
ax.set_title(f"{chosen_dye}  mol {cand['mol_index']}  CPs={cps[:-1]}")
plt.tight_layout()
plt.show()